# NOTEBOOK
# PARTE 1 - METRICHE DI CLASSIFICAZIONE
## PASSO 1 - SETUP E CLASSIFICATORE "SERIO"

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# 1. Caricamento MNIST
mnist = fetch_openml("mnist_784", as_frame=False, parser='auto')
X, y = mnist["data"], mnist["target"].astype(int)

# 2. Train/test split come da libro
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

# 3. Problema binario: 5 vs non-5
y_train_5 = (y_train == 5)
y_test_5 = (y_test == 5)

# 4. Classificatore lineare
sgd_clf = SGDClassifier(random_state=42)
y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

### Matrice di confusione

In [ ]:
cm = confusion_matrix(y_train_5, y_train_pred)
print(cm)

### Identificazione TN, FP, FN, TP

In [ ]:
TN = cm[0,0]
FP = cm[0,1]
FN = cm[1,0]
TP = cm[1,1]

### Calcolo di Precision, Recall, F1 e Accuracy

In [ ]:
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print(f"Precision Score: {precision:.2f}")
print(f"Recall Score: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

## PASSO 2 - DUMMY CLASSIFIER E ACCURACY INGANNEVOLE

In [ ]:
from sklearn.base import BaseEstimator
from sklearn.metrics import accuracy_score
import numpy as np

class NeverFiveClassifier(BaseEstimator):
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return np.zeros((len(X),), dtype=bool)

never_clf = NeverFiveClassifier()
y_train_pred_dummy = cross_val_predict(never_clf, X_train, y_train_5, cv=3)

### Matrice di confusione e metriche del dummy classifier

In [ ]:
cm_dummy = confusion_matrix(y_train_5, y_train_pred_dummy)
print(cm_dummy)

TN_d = cm_dummy[0,0]
FP_d = cm_dummy[0,1]
FN_d = cm_dummy[1,0]
TP_d = cm_dummy[1,1]

# Accuracy
acc_dummy = (TP_d + TN_d) / (TP_d + TN_d + FP_d + FN_d)
print(f"\nAccuracy: {acc_dummy:.3f}")

p_d = TP_d / (TP_d + FP_d) if (TP_d + FP_d) != 0 else 0.0
r_d = TP_d / (TP_d + FN_d) if (TP_d + FN_d) != 0 else 0.0
f1_d = 2 * p_d * r_d / (p_d + r_d) if (p_d + r_d) != 0 else 0.0

print(f"Precision Score: {p_d}")
print(f"Recall Score: {r_d}")
print(f"F1 Score: {f1_d}")

### Tabella di confronto

In [ ]:
import pandas as pd

confronto = pd.DataFrame({
    "Modello": ["SGD 5-detector", "Never-5"],
    "Accuracy": [f"{accuracy:.2f}", f"{acc_dummy:.2f}"],
    "Precision": [f"{precision:.2f}", f"{p_d:.2f}"],
    "Recall": [f"{recall:.2f}", f"{r_d:.2f}"],
    "F1": [f"{f1:.2f}", f"{f1_d:.2f}"]
})
confronto

### Perché l'accuracy è fuorviante

L'accuratezza risulta essere una metrica fortemente fuorviante quando si lavora con dataset sbilanciati, come nel problema di classificazione "5-vs-non-5". Poiché la classe positiva (il numero 5) rappresenta solo circa il 10% del totale delle immagini, un classificatore "banale" (dummy classifier) che prevede sistematicamente "non-5" otterrà un'accuratezza altissima, pari al 90%. Nonostante questo punteggio apparentemente eccellente, il modello è nei fatti inutile, dato che non riesce a identificare correttamente nemmeno un 5, restituendo infatti valori di Precision, Recall e F1 pari a zero. L'accuratezza maschera l'incapacità del modello di riconoscere la classe minoritaria; per questo motivo, in contesti sbilanciati, è fondamentale affidarsi alla matrice di confusione e a metriche più rappresentative.

# PARTE 2 - DECISION TREE SU IRIS
## PASSO 1 - Albero base su due feature (max_depth=2)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, export_graphviz
import graphviz
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

iris = load_iris()
X = iris.data[:, 2:]  # petal length, petal width
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

tree_clf = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_clf.fit(X_train, y_train)

### Visualizzazione dell'albero

In [ ]:
export_graphviz(
    tree_clf,
    out_file="iris_tree.dot",
    feature_names=iris.feature_names[2:],
    class_names=iris.target_names,
    rounded=True,
    filled=True
)

display(graphviz.Source.from_file("iris_tree.dot"))

### Screenshot dell'albero (max_depth=2)
![Decision Tree Iris](iris_decision_tree.jpg)

### Domande a risposta scritta

**1. Cosa rappresentano samples, value e gini in ogni nodo?**

`samples` indica il numero totale di istanze di training che raggiungono quel nodo. `value` è un array che mostra quante di queste istanze appartengono a ciascuna delle possibili classi di fiori. `gini` misura l'indice di impurità del nodo: vale 0 quando tutti i campioni appartengono a una sola classe e cresce quanto più le classi sono mescolate.

**2. Perché alcuni nodi hanno gini = 0?**

Il nodo è puro, ovvero tutti i samples che cadono in quel nodo appartengono alla stessa classe. Non è necessario effettuare ulteriori split perché la classificazione è già certa.

**3. Descrivi il percorso di decisione per classificare un'Iris come setosa.**

Partendo dal nodo radice, l'albero verifica la `petal length`. Se la lunghezza del petalo è inferiore o uguale a 2.45 cm, l'istanza segue il ramo di sinistra ("True") e viene classificata come setosa. Il nodo foglia risultante ha gini = 0, confermando che è un nodo puro.

## PASSO 2 - Esperimenti semplici
### Variare la profondità

#### max_depth=1

In [ ]:
tree_clf_1 = DecisionTreeClassifier(max_depth=1, random_state=42)
tree_clf_1.fit(X_train, y_train)

Si può notare che sia nel training che nel testing, l'accuracy è circa 0.67 (67%), quindi si ha **underfitting**. Il modello è troppo semplice: riesce a isolare correttamente solo la classe 0 (setosa) ma fallisce nel distinguere le classi 1 e 2.

#### max_depth=3

In [ ]:
tree_clf_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf_3.fit(X_train, y_train)

L'accuracy è molto alta: 0.98 (98%) nel training e 0.93 (93%) nel testing. Il modello risulta ben bilanciato, senza evidenti segni di underfitting o overfitting. L'aumento della profondità permette all'albero di creare regole sufficienti per distinguere con precisione tutte e tre le classi.

#### max_depth=None

In [ ]:
tree_clf_none = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_clf_none.fit(X_train, y_train)

Nel training l'accuracy è 0.99 (99%), ma nel testing rimane a 0.93 (93%). Si osserva **overfitting**: il modello è diventato più complesso senza migliorare le prestazioni su dati nuovi rispetto a `max_depth=3`.

### Gini vs Entropy (max_depth=3)

In [ ]:
tree_clf_gini = DecisionTreeClassifier(max_depth=3, random_state=42, criterion="gini")
tree_clf_gini.fit(X_train, y_train)

tree_clf_entropy = DecisionTreeClassifier(max_depth=3, random_state=42, criterion="entropy")
tree_clf_entropy.fit(X_train, y_train)

Confrontando i due criteri con `max_depth=3`, le prestazioni complessive sono uguali. Entrambi i modelli commettono in totale 2 errori sul training set e 3 errori sul test set, raggiungendo la stessa accuracy. L'unica differenza è nella distribuzione degli errori tra le classi 1 e 2. Ciò conferma che, per questo dataset, la scelta del criterio di impurità (Gini o Entropia) non ha un impatto significativo sulla capacità predittiva dell'albero.

### Uso di tutte le feature

In [ ]:
X_all = iris.data  # tutte e 4 le feature
y_all = iris.target

X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all)

tree_clf_all = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf_all.fit(X_train_all, y_train_all)

In [ ]:
export_graphviz(
    tree_clf_all,
    out_file="iris_tree_all.dot",
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    rounded=True,
    filled=True
)

display(graphviz.Source.from_file("iris_tree_all.dot"))

In questo caso la performance migliora: l'accuracy sul test set passa da 0.93 (93%) a 0.98 (98%). L'albero è ancora interpretabile, anche perché l'algoritmo ha effettuato una feature selection automatica, utilizzando solamente le misure dei petali e non quelle dei sepali.